In [5]:
import openmatrix as omx
import numpy as np
import os

import openmatrix
openmatrix.numpy = np

In [6]:
# update the input and output directories below to point to the correct locations
input_dir = r"C:\Users\USYS671257\WSP O365\Boston MPO Model Support - General\7. Next Generation Model Advancement\ActivitySim_Inputs\EXTERNAL\_skim"
output_dir = r"C:\Users\USYS671257\WSP O365\Boston MPO Model Support - General\7. Next Generation Model Advancement\ActivitySim_Inputs\skims"

In [7]:
skim_list = [
    {
        "name": "highway_am",
        "input_file_name": "hwy_am.omx",
        "output_file_name": "hwy_am_pm.omx",
        "tod": "am",
        "prefix": "",
        "matrices": ["da_time", "sr_time", "da_toll", "sr_toll", "dist", "rs_fare"],
        "clip_inf": True,
    },
    {
        "name": "highway_md",
        "input_file_name": "hwy_md.omx",
        "output_file_name": "hwy_md_ev.omx",
        "tod": "md",
        "prefix": "",
        "matrices": ["da_time", "sr_time", "da_toll", "sr_toll", "dist", "rs_fare"],
        "clip_inf": True,
    },
    {
        "name": "nm_daily",
        "input_file_name": "nm_daily.omx",
        "output_file_name": "nm_daily.omx",
        "tod": "nm",
        "prefix": "",
        "matrices": ["dist"],
        "clip_inf": True,
    },
    {
        "name": "tw_am",
        "input_file_name": "tw_am.omx",
        "output_file_name": "tw_am_pm.omx",
        "tod": "am",
        "prefix": "tw_",
        "matrices": ["xfer", "Fare", "iwait", "xwait", "ivtt", "walk", "gen_cost", "tdist"],
        "clip_inf": True,
    },
    {
        "name": "tw_md",
        "input_file_name": "tw_md.omx",
        "output_file_name": "tw_md_ev.omx",
        "tod": "md",
        "prefix": "tw_",
        "matrices": ["xfer", "Fare", "iwait", "xwait", "ivtt", "walk", "gen_cost", "tdist"],
        "clip_inf": True,
    },
    {
        "name": "ta_am",
        "input_file_name": "ta_am.omx",
        "output_file_name": "ta_am_pm.omx",
        "tod": "am",
        "prefix": "ta_",
        "matrices": ["xfer", "Fare", "iwait", "xwait", "ivtt", "walk", "dtime", "ddist", "tdist", "gen_cost", "auto_cost"],
        "clip_inf": True,
    },
    {
        "name": "ta_md",
        "input_file_name": "ta_md.omx",
        "output_file_name": "ta_md_ev.omx",
        "tod": "md",
        "prefix": "ta_",
        "matrices": ["xfer", "Fare", "iwait", "xwait", "ivtt", "walk", "dtime", "ddist", "tdist", "gen_cost", "auto_cost"],
        "clip_inf": True,
    },
]


In [8]:
for s in skim_list:
    input_file = os.path.join(input_dir, s["input_file_name"])
    output_file = os.path.join(output_dir, s["output_file_name"])

    matrix_in = omx.open_file(str(input_file), "r")
    shape = tuple(int(value) for value in matrix_in[s["matrices"][0]].shape)
    matrix_out = omx.open_file(str(output_file), "w", shape=shape)

    for mapping_name in matrix_in.list_mappings():
        if mapping_name in {"RCIndex", "Destination"}:
            old_mapping = matrix_in.mapping(mapping_name)
            old_index = [
                key for key, value in sorted(old_mapping.items(), key=lambda item: item[1])
            ]
            matrix_out.create_mapping("ID", old_index)
            break

    for matrix_name in s["matrices"]:
        print(f'{s["name"]}: process {matrix_name}')
        data = matrix_in[matrix_name][:]
        if s["clip_inf"]:
            data[(data < 0) | (data >= np.finfo(np.float32).max)] = 0

        base_name = f'{s["prefix"]}{matrix_name.lower()}'
        if s["tod"] == "nm":
            matrix_out["dist_nm"] = data
        elif s["tod"] == "am":
            matrix_out[f"{base_name}__AM"] = data
            matrix_out[f"{base_name}__PM"] = data.T
        else:
            matrix_out[f"{base_name}__MD"] = data
            matrix_out[f"{base_name}__EV"] = data.copy()

    if s["tod"] == "md" and s["prefix"] == "":
        matrix_out["DIST"] = matrix_out["dist__MD"][:]

    matrix_in.close()
    matrix_out.close()


highway_am: process da_time
highway_am: process sr_time
highway_am: process da_toll
highway_am: process sr_toll
highway_am: process dist
highway_am: process rs_fare
highway_md: process da_time
highway_md: process sr_time
highway_md: process da_toll
highway_md: process sr_toll
highway_md: process dist
highway_md: process rs_fare
nm_daily: process dist
tw_am: process xfer
tw_am: process Fare
tw_am: process iwait
tw_am: process xwait
tw_am: process ivtt
tw_am: process walk
tw_am: process gen_cost
tw_am: process tdist
tw_md: process xfer
tw_md: process Fare
tw_md: process iwait
tw_md: process xwait
tw_md: process ivtt
tw_md: process walk
tw_md: process gen_cost
tw_md: process tdist
ta_am: process xfer
ta_am: process Fare
ta_am: process iwait
ta_am: process xwait
ta_am: process ivtt
ta_am: process walk
ta_am: process dtime
ta_am: process ddist
ta_am: process tdist
ta_am: process gen_cost
ta_am: process auto_cost
ta_md: process xfer
ta_md: process Fare
ta_md: process iwait
ta_md: process xwa